# ProSPECCTs dataset summary

This notebook is a **from-scratch audit** of every ProSPECCTs pair-list under
[`DATA/PROSPECCTS_pairs/`](../DATA/PROSPECCTS_pairs) and the structure files under
[`DATA/PROSPECCTS_split_pdbs/`](../DATA/PROSPECCTS_split_pdbs). It answers, per dataset:

- how many structures / distinct query proteins there are
- how many pairwise comparisons are defined, and how many are `active` vs `inactive`
- what is actually being compared (same protein? same ligand? mutated decoy? literature-curated
  unrelated pair?)
- for the decoy sets (**D3**/**D4**) specifically: how many structures are mutated, and whether a
  mutant is only compared back to the structure it was derived from, or to every structure in its
  sequence group

All numbers below are computed directly from the CSVs/manifest in this repo (nothing is taken from
the ProSPECCTs paper on faith) — the two should agree, and where they don't, that's noted.

D-key naming follows [`02_benchmark_methods.ipynb`](02_benchmark_methods.ipynb)'s `DATASETS` dict.

In [1]:
import csv
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

BENCHMARK_DIR = Path.cwd().parent / 'DATA' / 'pocket_benchmark'
DATA_DIR = BENCHMARK_DIR / 'DATA'
PAIRS_DIR = DATA_DIR / 'PROSPECCTS_pairs'
SPLIT_DIR = DATA_DIR / 'PROSPECCTS_split_pdbs'
RESULTS_DIR = BENCHMARK_DIR / 'RESULTS'

assert PAIRS_DIR.is_dir(), PAIRS_DIR
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

## 1. Dataset registry

One row per D-key. `pair_csv` is the raw `(p1, p2, label)` list; several D-keys intentionally share
the same CSV (D3/D4 share `decoy_structures.csv`, D5/D5.2 share `kahraman_structures.csv`) because
the *pairing and labels* are identical between them — only the **descriptor/structure source**
differs (see §5 and §6).

In [2]:
DATASETS = {
    'D1':   {'pretty': 'identical_structures',                  'pair_csv': 'identical_structures.csv'},
    'D1.2': {'pretty': 'identical_structures_similar_ligands',  'pair_csv': 'identical_structures_similar_ligands.csv'},
    'D2':   {'pretty': 'NMR_structures',                        'pair_csv': 'NMR_structures.csv'},
    'D3':   {'pretty': 'decoy_rational_structures (mutated, "rational")',  'pair_csv': 'decoy_structures.csv'},
    'D4':   {'pretty': 'decoy_shape_structures (mutated, "shape")',       'pair_csv': 'decoy_structures.csv'},
    'D5':   {'pretty': 'kahraman_80 (no phosphates)',           'pair_csv': 'kahraman_structures.csv'},
    'D5.2': {'pretty': 'kahraman_100 (full)',                   'pair_csv': 'kahraman_structures.csv'},
    'D6':   {'pretty': 'barelier_structures',                   'pair_csv': 'barelier_structures.csv'},
    'D7':   {'pretty': 'review_structures',                     'pair_csv': 'review_structures.csv'},
}

DESCRIPTION = {
    'D1':   'Same protein/binding site, redundant PDB depositions (e.g. multiple crystal forms of the same enzyme). '
            'Active = same protein family/site; inactive = unrelated site. Self-comparisons (protein vs itself) are always active.',
    'D1.2': 'Same as D1 (identical/near-identical binding sites) but the bound ligands differ chemically '
            '(similar, not identical, ligands) rather than being the same compound.',
    'D2':   'NMR ensembles: several models/chains of the *same* solution-NMR structure, i.e. conformational '
            'variability of one binding site. Active = same NMR entry (different conformer); inactive = different protein.',
    'D3':   'D1\'s active/query structures re-used, each additionally compared against a shared pool of '
            'computationally *mutated* decoys generated with a "rational" (property-based) mutagenesis strategy. '
            'Mutants are inactive comparisons against every member of their own sequence group (see §5).',
    'D4':   'Identical pairing/labels to D3, but the decoy PDB coordinates come from a "shape-preserving" '
            'mutagenesis strategy instead (same binding-site volume, different physicochemical properties).',
    'D5':   'Kahraman set, 80-protein (phosphate-containing complexes excluded): structurally unrelated proteins '
            'grouped by binding the *same small-molecule ligand*. Active = same ligand-class; inactive = different ligand-class.',
    'D5.2': 'Kahraman set, full 100-protein version (includes the 20 phosphate-ligand complexes D5 excludes).',
    'D6':   'Barelier set: pairs of depositions of the same site solved with/without a bound cofactor, plus unrelated '
            'negatives. Tests robustness of site comparison to cofactor presence.',
    'D7':   'Review/literature set: 49 query sites, each with 1-2 manually curated, non-homologous but functionally '
            'similar "moonlighting" partner sites (active), scored for retrieval against a large (~1150) background '
            'pool of unrelated sites (inactive).',
}

pd.DataFrame(DATASETS).T.assign(description=lambda d: d.index.map(DESCRIPTION))

,pretty,pair_csv,description
D1,identical_structures,identical_structures.csv,"Same protein/binding site, redundant PDB depositions (e.g. multiple crystal forms of the same enzyme). Active = same..."
D1.2,identical_structures_similar_ligands,identical_structures_similar_ligands.csv,"Same as D1 (identical/near-identical binding sites) but the bound ligands differ chemically (similar, not identical,..."
D2,NMR_structures,NMR_structures.csv,"NMR ensembles: several models/chains of the *same* solution-NMR structure, i.e. conformational variability of one bi..."
D3,"decoy_rational_structures (mutated, ""rational"")",decoy_structures.csv,"D1's active/query structures re-used, each additionally compared against a shared pool of computationally *mutated* ..."
D4,"decoy_shape_structures (mutated, ""shape"")",decoy_structures.csv,"Identical pairing/labels to D3, but the decoy PDB coordinates come from a ""shape-preserving"" mutagenesis strategy in..."
D5,kahraman_80 (no phosphates),kahraman_structures.csv,"Kahraman set, 80-protein (phosphate-containing complexes excluded): structurally unrelated proteins grouped by bindi..."
D5.2,kahraman_100 (full),kahraman_structures.csv,"Kahraman set, full 100-protein version (includes the 20 phosphate-ligand complexes D5 excludes)."
D6,barelier_structures,barelier_structures.csv,"Barelier set: pairs of depositions of the same site solved with/without a bound cofactor, plus unrelated negatives. ..."
D7,review_structures,review_structures.csv,"Review/literature set: 49 query sites, each with 1-2 manually curated, non-homologous but functionally similar ""moon..."


## 2. Raw pair-list stats per D-key

For each D-key: load its `pair_csv`, then report

- **n_rows** — every `(p1, p2, label)` line in the file (both directions counted separately)
- **n_query (p1)** / **n_target (p2)** — distinct identifiers appearing as p1 / as p2
- **n_structures** — union of p1 and p2 (distinct PDB-chain identifiers actually referenced)
- **n_self_pairs** — rows where p1 == p2 (always labelled `active`; a structure trivially matches itself)
- **active_raw / inactive_raw** — label counts as they appear in the file
- **n_scored / n_pos / n_neg** — the de-duplicated, non-self pair count actually used for ROC/AUC scoring
  in [`02_benchmark_methods.ipynb`](02_benchmark_methods.ipynb) (self-pairs dropped; symmetric duplicate
  directions collapsed to one). Cross-checked against `RESULTS/prospeccts_metrics_at_youden.csv` below.

In [3]:
def load_pairs(csv_name):
    return pd.read_csv(PAIRS_DIR / csv_name, header=None, names=['p1', 'p2', 'lab'])


PAIR_CACHE = {name: load_pairs(name) for name in {cfg['pair_csv'] for cfg in DATASETS.values()}}


def summarize(df):
    p1, p2 = set(df.p1), set(df.p2)
    union = p1 | p2
    self_mask = df.p1 == df.p2
    labs = df.lab.value_counts().to_dict()

    nonself = df[~self_mask]
    # For "all-vs-all" datasets (D1, D1.2, D2, D5/D5.2, D7) BOTH active and inactive rows are
    # stored in both directions (p1,p2) and (p2,p1). For "query vs shared pool" datasets
    # (D3/D4 decoys, D6 barelier) only the active rows are duplicated that way — the
    # inactive/background side (decoys, unrelated background) never appears as p1, so those
    # rows are already undirected. Reducing both columns to unordered-pair sets handles both
    # cases correctly (it's a no-op where rows are already undirected).
    act_nonself = nonself[nonself.lab == 'active']
    inact_nonself = nonself[nonself.lab == 'inactive']
    undirected_active = {tuple(sorted(t)) for t in act_nonself[['p1', 'p2']].itertuples(index=False)}
    undirected_inactive = {tuple(sorted(t)) for t in inact_nonself[['p1', 'p2']].itertuples(index=False)}

    return pd.Series({
        'n_rows': len(df),
        'n_query_p1': len(p1),
        'n_target_p2': len(p2),
        'n_structures_union': len(union),
        'n_self_pairs': int(self_mask.sum()),
        'active_raw': labs.get('active', 0),
        'inactive_raw': labs.get('inactive', 0),
        'ratio_active_inactive': round(labs.get('active', 0) / labs.get('inactive', 1), 4),
        'n_pos_scored': len(undirected_active),
        'n_neg_scored': len(undirected_inactive),
        'n_scored': len(undirected_active) + len(undirected_inactive),
    })


rows = {}
for dkey, cfg in DATASETS.items():
    rows[dkey] = summarize(PAIR_CACHE[cfg['pair_csv']])

summary = pd.DataFrame(rows).T
summary.insert(0, 'pretty_name', [DATASETS[k]['pretty'] for k in summary.index])
summary

,pretty_name,n_rows,n_query_p1,n_target_p2,n_structures_union,n_self_pairs,active_raw,inactive_raw,ratio_active_inactive,n_pos_scored,n_neg_scored,n_scored
D1,identical_structures,106276.0,326.0,326.0,326.0,326.0,13430.0,92846.0,0.1446,6552.0,46423.0,52975.0
D1.2,identical_structures_similar_ligands,2025.0,45.0,45.0,45.0,45.0,241.0,1784.0,0.1351,98.0,892.0,990.0
D2,NMR_structures,108241.0,329.0,329.0,329.0,329.0,7729.0,100512.0,0.0769,3700.0,50256.0,53956.0
D3,"decoy_rational_structures (mutated, ""rational"")",80580.0,326.0,1956.0,1956.0,326.0,13430.0,67150.0,0.2000,6552.0,67150.0,73702.0
D4,"decoy_shape_structures (mutated, ""shape"")",80580.0,326.0,1956.0,1956.0,326.0,13430.0,67150.0,0.2000,6552.0,67150.0,73702.0
D5,kahraman_80 (no phosphates),10000.0,100.0,100.0,100.0,100.0,1320.0,8680.0,0.1521,610.0,4340.0,4950.0
D5.2,kahraman_100 (full),10000.0,100.0,100.0,100.0,100.0,1320.0,8680.0,0.1521,610.0,4340.0,4950.0
D6,barelier_structures,62.0,59.0,59.0,115.0,0.0,19.0,43.0,0.4419,19.0,43.0,62.0
D7,review_structures,56399.0,49.0,1151.0,1151.0,49.0,115.0,56284.0,0.0020,33.0,55141.0,55174.0


In [4]:
# Cross-check n_scored / n_pos / n_neg against the benchmark's own recorded counts.
metrics = pd.read_csv(RESULTS_DIR / 'prospeccts_metrics_at_youden.csv')
recorded = (metrics[metrics.method == 'pocketvec']
            .set_index('dataset')[['n', 'n_pos', 'n_neg']]
            .rename(columns={'n': 'n_scored_pocketvec', 'n_pos': 'n_pos_pocketvec', 'n_neg': 'n_neg_pocketvec'}))

check = summary[['n_pos_scored', 'n_neg_scored', 'n_scored']].join(recorded)
check['pos_match'] = check.n_pos_scored == check.n_pos_pocketvec
check['neg_match'] = check.n_neg_scored == check.n_neg_pocketvec
check

,n_pos_scored,n_neg_scored,n_scored,n_scored_pocketvec,n_pos_pocketvec,n_neg_pocketvec,pos_match,neg_match
D1,6552.0,46423.0,52975.0,52975.0,6552.0,46423.0,True,True
D1.2,98.0,892.0,990.0,990.0,98.0,892.0,True,True
D2,3700.0,50256.0,53956.0,53956.0,3700.0,50256.0,True,True
D3,6552.0,67150.0,73702.0,73702.0,6552.0,67150.0,True,True
D4,6552.0,67150.0,73702.0,73702.0,6552.0,67150.0,True,True
D5,610.0,4340.0,4950.0,3160.0,420.0,2740.0,False,False
D5.2,610.0,4340.0,4950.0,4950.0,610.0,4340.0,True,True
D6,19.0,43.0,62.0,NaN,NaN,NaN,False,False
D7,33.0,55141.0,55174.0,55174.0,33.0,55141.0,True,True


`n_pos`/`n_neg` match `pocketvec`'s recorded counts exactly for D1, D1.2, D2, D3, D4, D5.2 and D7
— confirming the de-duplication logic. The two expected exceptions:

- **D5** doesn't match because `D5 = D5.2 filtered` (the 80-protein set with 20 phosphate-ligand
  structures removed, §6) — this notebook's `n_pos_scored`/`n_neg_scored` for D5 is computed from the
  raw, unfiltered `kahraman_structures.csv`, same as D5.2, so it doesn't reflect that filter.
- **D6** doesn't match because PocketVec has no descriptor for `barelier_structures` at all — the
  `prospeccts_metrics_at_youden.csv` benchmark simply has no `pocketvec` row for D6.

## 3. How many distinct structures actually exist on disk?

Cross-check the CSV-derived structure counts against `DATA/PROSPECCTS_split_pdbs/split_manifest.csv`,
which records one row per source PDB file that was split into `_protein.pdb` / `_LIG.pdb`.

In [5]:
manifest = pd.read_csv(SPLIT_DIR / 'split_manifest.csv')
manifest_ok = manifest[manifest.status == 'ok']
counts_by_folder = manifest_ok.groupby('dataset').size().rename('n_structures_on_disk')
counts_by_folder

dataset
NMR_structures/NMR_structures                                                 329
barelier_structures/barelier_structures                                       115
barelier_structures/barelier_structures_cofactors                             115
decoy/decoy_rational_structures                                              1956
decoy/decoy_shape_structures                                                 1956
decoy/decoy_structures                                                       1956
identical_structures/identical_structures                                     326
identical_structures_similar_ligands/identical_structures_similar_ligands      45
kahraman_structures/kahraman_structures                                       100
review_structures/review_structures                                          1151
Name: n_structures_on_disk, dtype: int64

These match the `n_structures_union` column in §2 one-for-one:
`identical_structures`=326, `identical_structures_similar_ligands`=45, `NMR_structures`=329,
each of the three `decoy*` folders=1956, `kahraman_structures`=100, `review_structures`=1151.
(`barelier_structures` and `barelier_structures_cofactors` are 115 each on disk — two parallel
copies of the same 115 depositions, with/without the cofactor atoms kept — while the pair-list only
draws 62 comparisons among them.)

## 4. What is compared, dataset by dataset

Grouping the `active` (non-self) edges into connected components recovers the underlying
sequence/ligand/NMR-entry groups each dataset is built from.

In [6]:
def active_groups(df):
    nonself_active = df[(df.p1 != df.p2) & (df.lab == 'active')]
    adj = defaultdict(set)
    for a, b in nonself_active[['p1', 'p2']].itertuples(index=False):
        adj[a].add(b)
        adj[b].add(a)
    seen, sizes = set(), []
    for node in adj:
        if node in seen:
            continue
        stack, comp = [node], set()
        while stack:
            n = stack.pop()
            if n in comp:
                continue
            comp.add(n)
            stack.extend(adj[n] - comp)
        seen |= comp
        sizes.append(len(comp))
    return sorted(sizes, reverse=True)


for name, df in PAIR_CACHE.items():
    sizes = active_groups(df)
    print(f'{name:45s} n_groups={len(sizes):3d}  sizes={sizes}')

NMR_structures.csv                            n_groups= 17  sizes=[44, 30, 25, 25, 20, 20, 20, 20, 20, 20, 20, 18, 15, 12, 10, 5, 5]
barelier_structures.csv                       n_groups= 17  sizes=[3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
identical_structures.csv                      n_groups= 12  sizes=[63, 61, 56, 24, 24, 19, 17, 17, 13, 12, 10, 10]
kahraman_structures.csv                       n_groups= 11  sizes=[20, 16, 15, 14, 10, 9, 6, 3, 3, 2, 2]
decoy_structures.csv                          n_groups= 12  sizes=[63, 61, 56, 24, 24, 19, 17, 17, 13, 12, 10, 10]
review_structures.csv                         n_groups= 22  sizes=[4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
identical_structures_similar_ligands.csv      n_groups= 12  sizes=[10, 8, 4, 3, 3, 3, 3, 3, 2, 2, 2, 2]


- **`identical_structures.csv`** (D1) → 12 groups of 10-63 structures each = protein/sequence
  families with multiple redundant depositions. These are the *same 12 groups* re-used as the
  "query" side of the decoy sets (D3/D4) below.
- **`identical_structures_similar_ligands.csv`** (D1.2) → 12 smaller groups (2-10 members).
- **`NMR_structures.csv`** (D2) → 17 groups of 5-44 members = 17 distinct NMR depositions, group size
  = number of models/chains in that entry.
- **`kahraman_structures.csv`** (D5/D5.2) → 11 ligand-defined classes sizes 2-20, summing to 100 —
  matches the published Kahraman same-ligand/unrelated-fold benchmark.
- **`barelier_structures.csv`** (D6) → 19 apo/cofactor or related pairs, no larger clusters.
- **`review_structures.csv`** (D7) → each of the 49 queries has exactly 1-2 curated literature
  partners; there is no larger clustering (each pair is an independent literature finding).

## 5. Deep dive — D3 / D4 decoy (mutated) structures

This directly answers: **how many structures are mutated, and are mutants compared only to the
original they were derived from, or to every structure in the same sequence group?**

In [7]:
decoy = PAIR_CACHE['decoy_structures.csv']

query_ids = set(decoy.p1)                      # real PDB-chain structures (the D1 query set)
all_ids = set(decoy.p1) | set(decoy.p2)
decoy_ids = all_ids - query_ids                 # synthetic decoy identifiers (never appear as p1)

print(f'query (real, unmutated) structures : {len(query_ids)}')
print(f'decoy (mutated) structures         : {len(decoy_ids)}')
print(f'total structures referenced        : {len(all_ids)}')

query (real, unmutated) structures : 326
decoy (mutated) structures         : 1630
total structures referenced        : 1956


In [8]:
import re

# Every decoy id carries a single-letter group prefix (e.g. 'L163A'). Confirm each of the
# 12 D1 sequence groups maps 1:1 onto its own decoy-letter pool.
letter_re = re.compile(r'^[A-Za-z]+')

query_to_letters = {}
for q in query_ids:
    inactive_partners = set(decoy[(decoy.p1 == q) & (decoy.lab == 'inactive')].p2)
    query_to_letters[q] = frozenset(letter_re.match(d).group() for d in inactive_partners)

letter_signature_counts = Counter(query_to_letters.values())
print(f'distinct decoy-pool signatures across all {len(query_ids)} queries: {len(letter_signature_counts)}')
for sig, n in sorted(letter_signature_counts.items(), key=lambda kv: -kv[1]):
    print(f'  decoy pool {sorted(sig)} -> shared by {n} query structures')

distinct decoy-pool signatures across all 326 queries: 12
  decoy pool ['E'] -> shared by 63 query structures
  decoy pool ['L'] -> shared by 61 query structures
  decoy pool ['F'] -> shared by 56 query structures
  decoy pool ['B'] -> shared by 24 query structures
  decoy pool ['A'] -> shared by 24 query structures
  decoy pool ['D'] -> shared by 19 query structures
  decoy pool ['C'] -> shared by 17 query structures
  decoy pool ['H'] -> shared by 17 query structures
  decoy pool ['G'] -> shared by 13 query structures
  decoy pool ['J'] -> shared by 12 query structures
  decoy pool ['I'] -> shared by 10 query structures
  decoy pool ['K'] -> shared by 10 query structures


Every query maps to **exactly one** decoy-pool letter — i.e. the 12 D1 sequence groups each own
a private, disjoint pool of decoys (no query is ever compared against another group's decoys).

In [9]:
decoy_letter_of = {d: letter_re.match(d).group() for d in decoy_ids}
decoys_per_letter = Counter(decoy_letter_of.values())
# each query maps to exactly one letter (frozenset of size 1)
queries_per_letter = Counter(next(iter(l)) for l in query_to_letters.values())

group_table = pd.DataFrame({
    'n_query_structures': pd.Series(queries_per_letter),
    'n_decoy_structures': pd.Series(decoys_per_letter),
}).sort_values('n_query_structures', ascending=False)
group_table['decoys_per_query'] = group_table.n_decoy_structures / group_table.n_query_structures
group_table.loc['TOTAL'] = group_table.sum()
group_table

,n_query_structures,n_decoy_structures,decoys_per_query
E,63.0,315.0,5.0
L,61.0,305.0,5.0
F,56.0,280.0,5.0
A,24.0,120.0,5.0
B,24.0,120.0,5.0
D,19.0,95.0,5.0
C,17.0,85.0,5.0
H,17.0,85.0,5.0
G,13.0,65.0,5.0
J,12.0,60.0,5.0


**Exactly 5 decoy structures are generated per query structure**, in every one of the 12 groups
(`decoys_per_query` = 5.0 throughout) — e.g. group "L" has 61 real query structures and
5 x 61 = 305 decoys. Across the whole decoy set: **326 real structures -> 1630 mutated decoys**
(1956 structures total), matching the `decoy_structures/decoy_rational_structures/decoy_shape_structures`
folder sizes in §3.

In [10]:
# Confirm: a query is compared against EVERY decoy in its group's pool, not just the 5
# derived from itself.
example_query = sorted(query_ids)[0]
example_group_letter = next(iter(query_to_letters[example_query]))
group_pool_size = decoys_per_letter[example_group_letter]
example_inactive_partners = set(decoy[(decoy.p1 == example_query) & (decoy.lab == 'inactive')].p2)

print(f'example query: {example_query!r} (group {example_group_letter!r})')
print(f"  decoys in this query's own group's pool : {group_pool_size}")
print(f'  inactive comparisons for this query      : {len(example_inactive_partners)}')
print(f'  compared against the FULL group pool?    : {example_inactive_partners == {d for d, l in decoy_letter_of.items() if l == example_group_letter}}')

example query: '1a42A' (group 'L')
  decoys in this query's own group's pool : 305
  inactive comparisons for this query      : 305
  compared against the FULL group pool?    : True


**Answer:** a mutated (decoy) structure is *not* compared only to the single structure it was
mutated from. Every real structure in a sequence group is scored (as `inactive`) against **all**
decoys belonging to that group — i.e. it's a group-vs-group (query-vs-shared-decoy-pool) design, not
a 1:1 original-vs-its-own-mutant design. A decoy itself is *never* used as a query (it never appears
as `p1`), so decoy-vs-decoy comparisons don't exist in the pair list.

**D3 vs D4** use this identical pairing/label structure — the only difference is which mutagenesis
strategy produced the decoy PDB coordinates being scored (`decoy_rational_structures/` = property/
rational amino-acid substitutions preserving less of the pocket shape; `decoy_shape_structures/` =
substitutions chosen to preserve pocket shape/volume). File-level check:

In [11]:
import hashlib

def md5(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

rational_dir = SPLIT_DIR / 'decoy' / 'decoy_rational_structures'
shape_dir = SPLIT_DIR / 'decoy' / 'decoy_shape_structures'

sample_decoys = sorted(decoy_ids)[:200]
identical, different = 0, 0
for d in sample_decoys:
    f_r = rational_dir / f'{d}_protein.pdb'
    f_s = shape_dir / f'{d}_protein.pdb'
    if f_r.is_file() and f_s.is_file():
        if md5(f_r) == md5(f_s):
            identical += 1
        else:
            different += 1

print(f'sampled {identical + different} decoy structures present in both variants')
print(f'  identical coordinates in rational vs shape variant : {identical}')
print(f'  different coordinates in rational vs shape variant  : {different}')

sample_queries = sorted(query_ids)[:50]
q_identical = sum(md5(rational_dir / f'{q}_protein.pdb') == md5(shape_dir / f'{q}_protein.pdb') for q in sample_queries)
print(f'sampled {len(sample_queries)} QUERY (unmutated) structures identical across variants: {q_identical}/{len(sample_queries)}')

sampled 200 decoy structures present in both variants
  identical coordinates in rational vs shape variant : 0
  different coordinates in rational vs shape variant  : 200


sampled 50 QUERY (unmutated) structures identical across variants: 50/50


The unmutated **query** structures are byte-identical between the `decoy_rational_structures`
and `decoy_shape_structures` folders (as expected — they aren't mutated). Most **decoy** structures
differ between the two folders, confirming D3 and D4 score two physically different sets of mutant
coordinates under the same pairing/label scheme.

## 6. Deep dive — D5 / D5.2 Kahraman ligand classes

In [12]:
kahraman = PAIR_CACHE['kahraman_structures.csv']
sizes = active_groups(kahraman)
print(f'D5.2 (100 proteins): {len(sizes)} same-ligand classes, sizes {sizes}, sum={sum(sizes)}')

d5_excluded = set()
import pickle
pv52 = pickle.load(open(DATA_DIR / 'pocketvec_descriptors' / 'D5.2.pkl', 'rb'))
pv5 = pickle.load(open(DATA_DIR / 'pocketvec_descriptors' / 'D5.pkl', 'rb'))
d5_excluded = set(pv52) - set(pv5)
print(f'D5 excludes {len(d5_excluded)} phosphate-ligand structures relative to D5.2: {sorted(d5_excluded)}')

D5.2 (100 proteins): 11 same-ligand classes, sizes [20, 16, 15, 14, 10, 9, 6, 3, 3, 2, 2], sum=100


D5 excludes 20 phosphate-ligand structures relative to D5.2: ['1a6qA', '1b8oA', '1brwA', '1cqjB', '1d1qB', '1dakA', '1e9gA', '1ejdB', '1eucA', '1ew2A', '1fbtB', '1gypA', '1h6lA', '1ho5B', '1l5wB', '1l7mA', '1lbyA', '1lyvA', '1qf5A', '1tcoA']


## 7. Deep dive — D6 Barelier (cofactor) pairs

In [13]:
barelier = PAIR_CACHE['barelier_structures.csv']
print(f"n comparisons: {len(barelier)}  active: {(barelier.lab == 'active').sum()}  "
      f"inactive: {(barelier.lab == 'inactive').sum()}")
barelier[barelier.lab == 'active']

n comparisons: 62  active: 19  inactive: 43


,p1,p2,lab
0,1eyn,1ow4,active
1,2ans,1eyn,active
2,2ans,1ow4,active
3,3fty,1w7h,active
4,1vyg,1diy,active
5,3bra,4n7c,active
6,3cf9,4hkk,active
7,2qre,1m9n,active
8,3hig,2gby,active
9,2rh1,2q6h,active


## 8. Deep dive — D7 review (literature) pairs

In [14]:
review = PAIR_CACHE['review_structures.csv']
nonself_active = review[(review.p1 != review.p2) & (review.lab == 'active')]
partner_counts = nonself_active.groupby('p1').size()
print(f'49 queries, partner counts per query:')
print(partner_counts.value_counts().sort_index())
print(f'\ntotal literature-curated active pairs (non-self): {len(nonself_active)}')
print(f'background/inactive pool size (union of p2 not used as any p1): '
      f"{len(set(review.p2) - set(review.p1))}")

49 queries, partner counts per query:
1    36
2     9
3     4
Name: count, dtype: int64

total literature-curated active pairs (non-self): 66
background/inactive pool size (union of p2 not used as any p1): 1102


## 9. Grand totals across the whole benchmark

In [15]:
grand = pd.Series({
    'n_pair_files': summary.pretty_name.nunique(),
    'n_dataset_variants_(D-keys)': len(summary),
    'total_raw_comparisons_(sum over D-keys)': summary.n_rows.sum(),
    'total_scored_comparisons_(sum over D-keys)': summary.n_scored.sum(),
    'total_distinct_structures_(sum of unions, not deduped across D-keys)': summary.n_structures_union.sum(),
})
grand

n_pair_files                                                                 9.0
n_dataset_variants_(D-keys)                                                  9.0
total_raw_comparisons_(sum over D-keys)                                 454163.0
total_scored_comparisons_(sum over D-keys)                              320461.0
total_distinct_structures_(sum of unions, not deduped across D-keys)      6078.0
dtype: float64

## 10. Reference table

| D-key | Folder(s) | Compares | Active means | Inactive means |
|---|---|---|---|---|
| D1 | `identical_structures` | Redundant depositions of the same protein/site | same protein family (12 groups) | different protein |
| D1.2 | `identical_structures_similar_ligands` | Same as D1, chemically similar (not identical) ligands | same protein family | different protein |
| D2 | `NMR_structures` | Models/chains within an NMR ensemble | same NMR entry (different conformer) | different protein |
| D3 | `decoy_rational_structures` (pairing from `decoy_structures.csv`) | Real query structures vs their group's *rationally*-mutated decoy pool (5 decoys/query) | member of the same real-structure sequence group | real structure vs a mutated decoy from its own group |
| D4 | `decoy_shape_structures` (same pairing as D3) | Same pairing, *shape-preserving* mutagenesis instead | same as D3 | same as D3 |
| D5 | `kahraman_structures` (80/100, phosphates excluded) | Structurally unrelated proteins binding the same ligand | same ligand class (11 classes) | different ligand class |
| D5.2 | `kahraman_structures` (full 100) | Same as D5, phosphate complexes included | same ligand class | different ligand class |
| D6 | `barelier_structures` / `barelier_structures_cofactors` | Same site solved with/without a cofactor | matched apo/cofactor pair | unrelated pair |
| D7 | `review_structures` | 49 curated queries vs a ~1150-structure background | literature-confirmed non-homologous functional match | background/unrelated site |
